# Workshop 4 Lab — Advanced
### Computation 101 · IQ Biology Bootcamp 2026

This is the last lab, and it changes the job. You already know how to write bash, use git, and analyze
a real dataset in Python. Today you point an **AI assistant** at that same work — and learn the one skill
that makes AI safe for science: **you verify everything it hands you against something you already know.**

The data is the same **ZFP36L2 knockout** you analyzed in Workshop 3, and that's the whole trick: *you own
the answers.* When the assistant writes code, you don't hope it's right — you check its number against the
number you derived yourself. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your answer before running.

### First: open an AI assistant in another tab

This lab is **tool-agnostic** — any chat assistant works, and they're all free. Pick one:

- **ChatGPT** — chat.openai.com
- **Claude** — claude.ai
- **Google Gemini** — gemini.google.com (or aistudio.google.com)
- **Colab's own Gemini** — the **✦** button in the toolbar, or the Gemini side panel, right here in this window. Free for all Colab users, so there's nothing to install and no key to paste.

You'll copy a prompt from this notebook, paste it into your assistant, then bring the answer back here and
**run it against the ground truth.** No API keys, no setup — you are the bridge between the two tabs.

In [ ]:
# One-time setup. The line starting with ! is a REAL shell command in this Colab Linux
# machine — the same git you used on Fiji. It clones the course repo (only if it isn't
# here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Drive the assistant — and check its work at every step
Workshop 3 only looked at genes going **up**. Now use an assistant to push the analysis into new territory —
the **down**-regulated side, and a full six-tissue picture — verifying each result against a file in the repo
before you believe it. This is real: you're computing things the Workshop 3 lab never showed you.

## Step 1 · The down-regulated side
Ask your assistant:

> Write `count_down(path)` like a DESeq2 down-regulation filter: genes with `log2FoldChange < -1` and
> `padj < 0.05`. Read the CSV with pandas, gene IDs in column 0, return the count.

Then verify it against a file the repo already carries: `derived/BM_down_geneids.txt` is the exact list of
bone-marrow down genes. Its length is your ground truth — no need to trust the function's number blind.

In [ ]:
def count_down(path):
    df = pd.read_csv(path, index_col=0)
    hits = df[( ... ) & (df['padj'] < 0.05)]   # down-regulated: log2FoldChange < -1
    return len(hits)

n = count_down(os.path.join(DATA, 'de', 'BM_DE.csv'))
known = len(open(os.path.join(DATA, 'derived', 'BM_down_geneids.txt')).read().split())
print(f'function: {n}   repo file: {known}   →', 'MATCH ✓' if n == known else 'MISMATCH ✗')

Verifying against a **second, independent source** (the derived file, not just a number you remember) is the
strongest check you have. If the function and the file disagree, you found a bug before it found you.

## Step 2 · The whole map, both directions
Now ask the assistant for one function that returns **up and down counts for every tissue** as a table:

> Write `updown_table(data_dir, tissues)` that, for each tissue, reads `{tissue}_DE.csv` and returns a pandas
> DataFrame with columns `up` (log2FC > 1, padj < 0.05) and `down` (log2FC < -1, padj < 0.05), one row per tissue.

Fill in the two filters, then check the table against the counts you already know.

In [ ]:
tissues = ['Lung', 'Liver', 'BM', 'Spleen', 'Ovary', 'Kidney']
def updown_table(data_dir, tissues):
    rows = {}
    for t in tissues:
        df = pd.read_csv(os.path.join(data_dir, 'de', f'{t}_DE.csv'), index_col=0)
        up   = (df['log2FoldChange'] > 1)  & (df['padj'] < 0.05)
        down = ( ... ) & (df['padj'] < 0.05)   # the down filter: log2FoldChange < -1
        rows[t] = {'up': int(up.sum()), 'down': int(down.sum())}
    return pd.DataFrame(rows).T

tbl = updown_table(DATA, tissues)
print(tbl)

In [ ]:
# Verify the whole table at once against the counts you trust.
expected_up = {'Lung': 71, 'Liver': 404, 'BM': 1343, 'Spleen': 573, 'Ovary': 291, 'Kidney': 53}
ok = all(int(tbl.loc[t, 'up']) == expected_up[t] for t in expected_up)
print('all six up-counts match Workshop 3:', 'YES ✓' if ok else 'NO ✗')

**Notice Spleen** — the only tissue with *more* genes going down (657) than up (573). You just found a real
asymmetry in the data that the up-only Workshop 3 lab never revealed, and you trust it because every number
checked out.

### Two habits that made this work
- **Prompt hygiene** — you named the columns (`log2FoldChange`, `padj`), gave the exact thresholds, and asked
  for *one* function, not a paragraph of vague intent. Precise prompts get precise code.
- **Demand rewrites, don't patch** — if the assistant's version had come back tangled, the move isn't to nudge
  it line by line. Say: *'knowing what you know now, scrap it and write the clean version.'* The rewrite is
  almost always better than the patch.

### Try next (no answer key — you're past that)
Ask your assistant to compute: of bone marrow's down-regulated genes, how many carry an eCLIP 3′UTR footprint
(`derived/eclip_3utr_genes.txt`)? Then verify its logic by reading the code, not by trusting the number.
The **Expert** workbook flips the order entirely: write the test *first*, then let AI write code to pass it.

## One more thing

You just used AI the way a scientist has to — as a fast, confident junior colleague whose every claim you
check before you trust it. And you *could* check it, because the ground truth was real: a published study,
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program a year ago, sitting where you are now.
Across four workshops you took one real dataset from a raw shell prompt to a version-controlled, Python-analyzed,
AI-assisted result: **bash → git → Python → AI**, every step on the same science. The assistant made you faster.
Owning the answer is what kept you honest. That's the job now — and you can already do it.